# Qwen3Guard proxy observable and suffix SMC smoke
This notebook runs infrastructure checks only. It does not estimate a rare-event probability and does not use the unpublished StreamGuard checkpoint.

In [ ]:
# EDIT ONLY THESE SMOKE PARAMETERS
REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git'
REPO_DIR = '/content/rea_of_llms_supplementary_code-main'
GENERATOR_MODEL = 'Qwen/Qwen3-1.7B'
GENERATOR_DEVICE = 'cuda'
GUARD_DEVICE = 'cuda'
DTYPE = 'float16'
SEED = 7319

In [ ]:
from pathlib import Path
import os, subprocess
repo = Path(REPO_DIR)
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
os.chdir(repo)
print('cwd =', Path.cwd())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'qwen_safety_smc_experiment/requirements.txt'], check=True)

In [ ]:
import torch
print('torch:', torch.__version__)
print('compiled CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if GENERATOR_DEVICE == 'cuda' or GUARD_DEVICE == 'cuda':
    assert torch.cuda.is_available(), 'CUDA was requested but is unavailable'
    print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 1024**3)

In [ ]:
import subprocess, sys
command = [
    sys.executable, 'qwen_safety_smc_experiment/scripts/run_smoke.py',
    '--generator-model', GENERATOR_MODEL,
    '--generator-device', GENERATOR_DEVICE,
    '--guard-device', GUARD_DEVICE,
    '--dtype', DTYPE,
    '--seed', str(SEED),
]
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json
from IPython.display import display, Image
runs = sorted(Path('qwen_safety_smc_experiment/outputs').glob('smoke_*'))
latest = runs[-1]
print('Latest smoke output:', latest)
print((latest / 'smoke.log').read_text()[-6000:])
display(Image(filename=str(latest / 'prefix_trajectory.png')))
display(json.loads((latest / 'smoke_summary.json').read_text()))